# LangChain Fundamentals

Install Dependencies

Versions are pinned to LangChain 0.3 (same as our LangGraph notebook). The latest LangChain 1.x removed classes like `LLMChain` and `ConversationChain` used here.

In [ ]:
!pip install -q langchain==0.3.25 langchain-core==0.3.60 langchain-community==0.3.24 langchain-openai==0.3.15 openai==1.82.0 google-search-results numexpr wikipedia

Connect to LLM

`ChatOpenAI` works with any OpenAI-compatible server. Here we use OpenRouter (free models). For LM Studio, change `base_url` to `http://localhost:1234/v1`.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(base_url="https://openrouter.ai/api/v1", api_key="api-key", model="nvidia/nemotron-3.5-lightning:free", temperature=0.9) #max_tokens=100

In [2]:
query = "What would be a good company name for a company that makes colorful socks?"

Basic Run

`invoke()` sends a message to the model and returns an `AIMessage`. The text is in `.content`.

In [ ]:
response = llm.invoke([HumanMessage(content=query)])
print(response)

Prompt Template

A reusable prompt with `{placeholders}` that we fill using `.format()`.

In [ ]:
from langchain_core.prompts import PromptTemplate

template = "I want to open a restaurant for {cuisine} food. Suggest a fency name for this."

prompt = PromptTemplate(
    input_variables=['cuisine'],
    template=template
)

formatted_prompt = prompt.format(cuisine="Chinese")
print(formatted_prompt)

Few Shot Prompts

We give the model a few examples first, so it follows the same format for the new word.

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate


examples = [
    {"word": "cat", "definition": "A small domesticated carnivorous mammal."},
    {"word": "dog", "definition": "A domesticated carnivorous mammal that typically has a long snout."},
]

example_prompt = PromptTemplate(
    input_variables=["word", "definition"],
    template="Word: {word}\nDefinition: {definition}\n"
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Keep the answer very short and consise and Define the following words:",
    suffix="Word: {input}\nDefinition:",
    input_variables=["input"],
)

output = (few_shot_prompt.format(input="elephant"))
print (output)

In [ ]:
output = llm.invoke([HumanMessage(content=output)])
print(output.content)

Simple Chain

A chain connects a prompt and an LLM: input → prompt → LLM → output.

In [ ]:
from langchain.chains import LLMChain

prompt = PromptTemplate(
    input_variables=["topic"],
    template="Give me a tweet idea about {topic}"
)

chain = LLMChain(llm=llm, prompt=prompt)

response = chain.run("AI in education")
print(response)

Sequential Chain

Runs chains one after another. The output of the first chain (`title`) becomes the input of the second.

In [ ]:
from langchain.chains import SequentialChain

# Prompt 1: Generate title
title_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Write a compelling blog post title about {topic}. only give one one title as the output nothing more than that"
)
title_chain = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

# Prompt 2: Use title to generate intro
intro_prompt = PromptTemplate(
    input_variables=["title"],
    template="Write a short blog introduction for the post titled: {title}. only give the introduction as the output nothing more than that. have bullet points and emojis in the intro"
)
intro_chain = LLMChain(llm=llm, prompt=intro_prompt, output_key="intro")

# Chain them together
overall_chain = SequentialChain(
    chains=[title_chain, intro_chain],
    input_variables=["topic"],
    output_variables=["title", "intro"]
)

result = overall_chain.invoke("LangChain for AI-powered apps")
print(result)

In [ ]:
print(result["intro"])

Memory

LLMs don't remember previous messages. `ConversationBufferMemory` stores the chat and adds it to every new prompt. `verbose=True` shows the full prompt, so you can see the history growing.

In [ ]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True,  # shows the input/output/logs
)

print (conversation.predict(input="Hi, my name is Deepan."))


print (conversation.predict(input="What’s my name?"))

Custom Parser

An output parser cleans the model's response. This one removes `<think>...</think>` blocks that reasoning models (like Qwen3) produce.

In [ ]:
import re
from langchain_core.output_parsers import BaseOutputParser

memory = ConversationBufferMemory()

class RemoveThinkParser(BaseOutputParser):
    def parse(self, text: str) -> str:
        # remove <think>...</think> blocks
        return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True,
    output_parser=RemoveThinkParser()
)

print (conversation.predict(input="Hi, my name is Deepan."))


print (conversation.predict(input="What’s my name?"))

Custom Prompt template

Our own prompt for the conversation. It must have `{history}` (filled by memory) and `{input}` (the user message).

In [ ]:
memory = ConversationBufferMemory()

custom_prompt = PromptTemplate(
    input_variables=["history", "input"],
    template="""
You are a helpful and concise assistant.

{history}
Human: {input}
AI:"""
)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True,
    prompt=custom_prompt,
    output_parser=RemoveThinkParser()
)


print (conversation.predict(input="Hi, my name is Deepan."))


print (conversation.predict(input="What’s my name?"))

Conversation Summary Memory

Instead of storing the full chat, the LLM keeps a short summary of it. This saves tokens in long conversations.

In [ ]:
from langchain.memory import ConversationSummaryMemory

summary_memory = ConversationSummaryMemory(llm=llm)

conversation = ConversationChain(
    llm=llm,
    memory=summary_memory,
    verbose=False,
    prompt=custom_prompt,
    output_parser=RemoveThinkParser()
)


print (conversation.predict(input="Hi, my name is Deepan."))

print (conversation.predict(input="i want to learn about AI and Machine learning."))

print (conversation.predict(input="how can you help me?"))

print (conversation.predict(input="What’s my name?"))

Memory with Chains

Memory can be added to any chain. The prompt needs `{history}`, otherwise the model never sees the previous messages.

In [ ]:
prompt = PromptTemplate(
    input_variables=["history", "input"],
    template="{history}\nThe user said: {input}"
)

chain_with_memory = LLMChain(
    llm=llm,
    prompt=prompt,
    memory=ConversationBufferMemory()
)

print(chain_with_memory.predict(input="Hello there!"))
print(chain_with_memory.predict(input="What did I just say?"))